# Data Read

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import anndata as ad
from scipy import sparse



## Function

In [12]:
from scipy.io import mmread
from scipy import sparse
import pandas as pd
import anndata as ad
import numpy as np
import os
import glob


def read_atac_auto(data_dir, sample_key="GM12878.rep2", genome=None):
    
    if genome:
        count_pattern = f"*{sample_key}*{genome}*.counts.txt.gz"
        peaks_pattern = f"*{sample_key}*{genome}*.peaks.bed.gz"
        barcodes_pattern = f"*{sample_key}*{genome}*.barcodes.txt.gz"
        fragment_pattern = f"*{sample_key}*{genome}*.fragments.bed.gz"
    else:
        count_pattern = f"*{sample_key}*.counts.txt.gz"
        peaks_pattern = f"*{sample_key}*.peaks.bed.gz"
        barcodes_pattern = f"*{sample_key}*.barcodes.txt.gz"
        fragment_pattern = f"*{sample_key}*.fragments.bed.gz"
        
    all_counts = glob.glob(os.path.join(data_dir, "**", count_pattern), recursive=True)
    
    counts_files = [f for f in all_counts if ".rna." not in os.path.basename(f)]

    print("*="*20)
    print(count_pattern)
    
    if len(counts_files) == 0:
        raise FileNotFoundError(f"Cannot find ATAC counts for {sample_key} (genome={genome})")
    if len(counts_files) > 1:
        print("Matched multiple ATAC count files:")
        for f in counts_files:
            print("  ", f)
        print("Use first one.")

    counts_path = counts_files[0]
    peaks_path = find_one(data_dir, peaks_pattern)
    barcodes_path = find_one(data_dir, barcodes_pattern)

    fragment_candidates = glob.glob(
        os.path.join(data_dir, "**", fragment_pattern),
        recursive=True
    )
    fragments_path = fragment_candidates[0] if fragment_candidates else None

    print("ATAC counts:", counts_path)
    print("ATAC peaks:", peaks_path)
    print("ATAC barcodes:", barcodes_path)
    print("ATAC fragment:", fragments_path)

    # 1. 读取 peaks
    peaks = pd.read_csv(
        peaks_path,
        sep="\t",
        header=None,
        compression="gzip"
    ).iloc[:, :3]

    peaks.columns = ["chrom", "start", "end"]
    peaks["peak_id"] = (
        peaks["chrom"].astype(str)
        + ":"
        + peaks["start"].astype(str)
        + "-"
        + peaks["end"].astype(str)
    )

    # 2. 读取 barcodes
    barcodes = pd.read_csv(
        barcodes_path,
        sep="\t",
        header=None,
        compression="gzip"
    )[0].astype(str).values

    n_peaks = peaks.shape[0]
    n_cells = len(barcodes)

    print("n_peaks:", n_peaks)
    print("n_cells:", n_cells)

    # 3. 读取 MatrixMarket 稀疏矩阵
    # 原始矩阵是 peak × cell
    mat = mmread(counts_path).tocsr()

    print("Raw ATAC matrix shape from mtx:", mat.shape)

    if mat.shape != (n_peaks, n_cells):
        raise ValueError(
            f"Matrix shape {mat.shape} does not match peaks/cells: "
            f"({n_peaks}, {n_cells})"
        )

    # AnnData 需要 cell × peak
    X = mat.T.tocsr().astype(np.float32)

    # 4. 构造 AnnData
    atac = ad.AnnData(X=X)

    atac.obs_names = barcodes.astype(str)
    atac.var_names = peaks["peak_id"].astype(str).values

    atac.var["chrom"] = peaks["chrom"].values
    atac.var["start"] = peaks["start"].astype(int).values
    atac.var["end"] = peaks["end"].astype(int).values

    atac.obs["sample"] = sample_key
    atac.layers["counts"] = atac.X.copy()

    if fragments_path is not None:
        atac.uns["files"] = {"fragments": fragments_path}

    atac.obs_names_make_unique()
    atac.var_names_make_unique()

    print("ATAC AnnData:", atac)
    return atac

def find_one(data_dir, pattern):
    files = glob.glob(os.path.join(data_dir, "**", pattern), recursive=True)

    if len(files) == 0:
        raise FileNotFoundError(f"Cannot find file pattern: {pattern}")

    if len(files) > 1:
        print("Matched multiple files:")
        for f in files:
            print("  ", f)
        print("Use first one.")

    return files[0]

def read_rna_auto(data_dir, sample_key="GM12878.rep2", genome=None):
    
     # RNA 文件格式: GM12878.3T3.rna.hg19.counts.txt.gz
    if genome:
        rna_pattern = f"*{sample_key}*.rna*{genome}*.counts.txt.gz"
    else:
        rna_pattern = f"*{sample_key}*.rna*.counts.txt.gz"

    rna_files = glob.glob(os.path.join(data_dir, "**", rna_pattern), recursive=True)

    if len(rna_files) == 0:
        raise FileNotFoundError(f"Cannot find RNA counts for {sample_key} (genome={genome})")
    if len(rna_files) > 1:
        print("Matched multiple RNA files:")
        for f in rna_files:
            print("  ", f)
        print("Use first one.")

    rna_path = rna_files[0]
    
    # 检查文件是否为空
    if os.path.getsize(rna_path) == 0:
        print(f"RNA file is empty (0 bytes): {rna_path}")
        return ad.AnnData()

    print("RNA counts:", rna_path)


    # 关键：必须显式指定 tab 分隔
    counts = pd.read_csv(
        rna_path,
        sep="\t",
        compression="gzip",
        header=0,
        index_col=0
    )

    print("RNA raw shape:", counts.shape)
    print(counts.iloc[:5, :5])

    # 全部转成数值，无法转换的设为 NaN
    counts = counts.apply(pd.to_numeric, errors="coerce")

    # 检查 NaN
    n_nan = counts.isna().sum().sum()
    print(f"Total NaN in RNA matrix: {n_nan}")

    # 如果有 NaN，通常说明文件某些位置仍有非数字字符
    # 对 count matrix 来说，直接补 0 更稳
    if n_nan > 0:
        counts = counts.fillna(0)

    # gene × cell -> cell × gene
    X = sparse.csr_matrix(counts.T.values.astype(np.float32))

    rna = ad.AnnData(X=X)
    rna.obs_names = counts.columns.astype(str)
    rna.var_names = counts.index.astype(str)

    rna.obs["sample"] = sample_key
    rna.layers["counts"] = rna.X.copy()

    rna.obs_names_make_unique()
    rna.var_names_make_unique()

    print("RNA AnnData:", rna)
    return rna

import re

def paired_adata(adata_rna, adata_atac):
    """SHARE-seq 数据配对：自动识别两种 barcode 格式"""
    
    def get_cell_key(barcode):
        if "," in barcode:
            # 格式1: R1.33,R2.01,R3.47,P1.44
            parts = barcode.split(",")
            return ",".join(parts[:3])  # R1.33,R2.01,R3.47
        else:
            # 格式2: R1.01.R2.01.R3.06.P1.55
            # 用正则提取 R1.XX, R2.XX, R3.XX，去掉 P1.XX
            match = re.match(r'(R1\.\d+)\.(R2\.\d+)\.(R3\.\d+)\.P1\.\d+', barcode)
            if match:
                return f"{match.group(1)},{match.group(2)},{match.group(3)}"
            else:
                # 兜底：去掉最后一个 .P1.XX
                parts = barcode.rsplit('.P1.', maxsplit=1)
                key = parts[0]
                # 统一为逗号分隔格式
                key = re.sub(r'\.(R[23]\.)', r',\1', key)
                return key
    
    rna = adata_rna.copy()
    atac = adata_atac.copy()
    
    # 检测格式
    sample_rna = str(rna.obs_names[0])
    sample_atac = str(atac.obs_names[0])
    print(f"RNA  barcode 示例: {sample_rna}")
    print(f"ATAC barcode 示例: {sample_atac}")
    print(f"RNA  格式: {'逗号' if ',' in sample_rna else '点'}")
    print(f"ATAC 格式: {'逗号' if ',' in sample_atac else '点'}")
    
    rna.obs["cell_key"] = [get_cell_key(str(bc)) for bc in rna.obs_names]
    atac.obs["cell_key"] = [get_cell_key(str(bc)) for bc in atac.obs_names]
    
    # 验证提取结果
    print(f"\nCell key 示例:")
    print(f"  RNA:  {rna.obs_names[0]} → {rna.obs['cell_key'].iloc[0]}")
    print(f"  ATAC: {atac.obs_names[0]} → {atac.obs['cell_key'].iloc[0]}")
    
    # 去重
    rna_dup = rna.obs["cell_key"].duplicated(keep="first")
    atac_dup = atac.obs["cell_key"].duplicated(keep="first")
    print(f"\nRNA  duplicates: {rna_dup.sum()}")
    print(f"ATAC duplicates: {atac_dup.sum()}")
    
    rna = rna[~rna_dup].copy()
    atac = atac[~atac_dup].copy()
    
    # 找交集
    common_keys = sorted(set(rna.obs["cell_key"]) & set(atac.obs["cell_key"]))
    print(f"\nRNA  cells: {rna.n_obs}")
    print(f"ATAC cells: {atac.n_obs}")
    print(f"Paired cells: {len(common_keys)}")
    
    if len(common_keys) == 0:
        print("\n[警告] 0 个配对！检查 cell key:")
        print(f"  RNA  key 前5: {rna.obs['cell_key'].head().tolist()}")
        print(f"  ATAC key 前5: {atac.obs['cell_key'].head().tolist()}")
        raise ValueError("无法配对，barcode 格式不匹配")
    
    # 对齐
    rna.obs.index = rna.obs["cell_key"]
    atac.obs.index = atac.obs["cell_key"]
    
    rna = rna[common_keys, :].copy()
    atac = atac[common_keys, :].copy()
    
    assert (rna.obs_names == atac.obs_names).all(), "配对失败！"
    print("配对成功！")
    
    return rna, atac

## Data read

In [17]:

data_dir = "/home/wuyan/dygmamba_project/NewRealPlan/case3/data/GSE140203"
out_dir = "/home/wuyan/dygmamba_project/NewRealPlan/case3/data/GSE140203/h5ad"
os.makedirs(out_dir, exist_ok=True)

# 'GM12878.rep1',
sample_configs = [
    ('GM12878.rep1',       None),
    ('GM12878.rep2',       None),
    ('GM12878.3T3',        'hg19'),
    ('GM12878.3T3',        'mm10'),
    ('K562.Raw',           'hg19'),
    ('K562.Raw',           'mm10'),
    ('skin.late.anagen',   None),
    ('brain',              None),
    ('lung',               None),
]

rna_list = []
atac_list = []

for sample_key, genome in sample_configs:
    
    print("*" * 50)
    tag = f"{sample_key}.{genome}" if genome else sample_key
    print(tag)

    atac = read_atac_auto(data_dir, sample_key, genome)
    
    if len(atac) == 0:
        print(f"{sample_key} has no ATAC file")
        continue
    
    rna = read_rna_auto(data_dir, sample_key, genome)
    
    if len(rna) == 0:
        print(f"{sample_key} has no RNA file")
        continue
        
    
    atac.write_h5ad(os.path.join(out_dir, tag + "_atac.h5ad"))
    rna.write_h5ad(os.path.join(out_dir, tag + "_rna.h5ad"))

    print("Saved:")
    print(os.path.join(out_dir, tag + "_atac.h5ad"))
    print(os.path.join(out_dir, tag + "_rna.h5ad"))

**************************************************
GM12878.rep1
*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=
*GM12878.rep1*.counts.txt.gz
ATAC counts: /home/wuyan/dygmamba_project/NewRealPlan/case3/data/GSE140203/GSM4156590_GM12878.rep1.counts.txt.gz
ATAC peaks: /home/wuyan/dygmamba_project/NewRealPlan/case3/data/GSE140203/GSM4156590_GM12878.rep1.peaks.bed.gz
ATAC barcodes: /home/wuyan/dygmamba_project/NewRealPlan/case3/data/GSE140203/GSM4156590_GM12878.rep1.barcodes.txt.gz
ATAC fragment: /home/wuyan/dygmamba_project/NewRealPlan/case3/data/GSE140203/GSM4156590_GM12878.rep1.atac.fragments.bed.gz
n_peaks: 507307
n_cells: 1932
Raw ATAC matrix shape from mtx: (507307, 1932)
ATAC AnnData: AnnData object with n_obs × n_vars = 1932 × 507307
    obs: 'sample'
    var: 'chrom', 'start', 'end'
    uns: 'files'
    layers: 'counts'
RNA file is empty (0 bytes): /home/wuyan/dygmamba_project/NewRealPlan/case3/data/GSE140203/GSM4156601_GM12878.rep1.rna.counts.txt.gz
GM12878.rep1 has no RNA file
*********

# Paired Data

In [13]:
import anndata as ad
import os

out_dir = "/home/wuyan/dygmamba_project/NewRealPlan/case3/data/GSE140203/h5ad"

data_path = "/home/wuyan/dygmamba_project/NewRealPlan/case3/data/process/"

sample_configs = [
    ('GM12878.rep1',       None),
    ('GM12878.rep2',       None),
    ('GM12878.3T3',        'hg19'),
    ('GM12878.3T3',        'mm10'),
    ('K562.Raw',           'hg19'),
    ('K562.Raw',           'mm10'),
    ('skin.late.anagen',   None),
    ('brain',              None),
    ('lung',               None),
]

rna_list = []
atac_list = []

for sample_key, genome in sample_configs:
    tag = f"{sample_key}.{genome}" if genome else sample_key
    
    rna_path  = os.path.join(out_dir, tag + "_rna.h5ad")
    atac_path = os.path.join(out_dir, tag + "_atac.h5ad")
    
    if not os.path.exists(rna_path) or not os.path.exists(atac_path):
        print(f"[跳过] {tag}: 文件不存在")
        continue
    
    print(f"\n{'='*50}")
    print(f"处理: {tag}")
    
    rna  = ad.read_h5ad(rna_path)
    atac = ad.read_h5ad(atac_path)
    
    print(f"  配对前 - RNA: {rna.n_obs}, ATAC: {atac.n_obs}")
    
    # 配对
    try:
        rna, atac = paired_adata(rna, atac)
    except Exception as e:
        print(f"[跳过] {tag} 配对失败: {e}")
        continue
    
    if rna.n_obs == 0:
        print(f"[跳过] {tag}: 配对后无细胞")
        continue
    
    # 添加细胞类型标注
    rna.obs["cell_type"]  = tag
    atac.obs["cell_type"] = tag
    
    # 确保 obs_names 全局唯一（加前缀）
    rna.obs_names  = [f"{tag}_{bc}" for bc in rna.obs_names]
    atac.obs_names = [f"{tag}_{bc}" for bc in atac.obs_names]
    
    rna_list.append(rna)
    atac_list.append(atac)
    
    print(f"  配对后: {rna.n_obs} 细胞")

# 整合
print(f"\n{'='*50}")
print(f"共 {len(rna_list)} 个样本成功配对")

adata_rna  = ad.concat(rna_list,  join="outer", fill_value=0)
adata_atac = ad.concat(atac_list, join="outer", fill_value=0)

print(f"整合后 RNA:  {adata_rna.shape}")
print(f"整合后 ATAC: {adata_atac.shape}")
print(f"细胞类型分布:")
print(adata_rna.obs["cell_type"].value_counts())

# 保存
adata_rna.write_h5ad(os.path.join(data_path, "merged_rna.h5ad"))
adata_atac.write_h5ad(os.path.join(data_path, "merged_atac.h5ad"))
print("保存完成")

[跳过] GM12878.rep1: 文件不存在

处理: GM12878.rep2
  配对前 - RNA: 2972, ATAC: 3621
RNA  barcode 示例: R1.01,R2.01,R3.03,P1.40
ATAC barcode 示例: R1.01,R2.01,R3.21,P1.33
RNA  格式: 逗号
ATAC 格式: 逗号

Cell key 示例:
  RNA:  R1.01,R2.01,R3.03,P1.40 → R1.01,R2.01,R3.03
  ATAC: R1.01,R2.01,R3.21,P1.33 → R1.01,R2.01,R3.21

RNA  duplicates: 29
ATAC duplicates: 38

RNA  cells: 2943
ATAC cells: 3583
Paired cells: 2788
配对成功！
  配对后: 2788 细胞

处理: GM12878.3T3.hg19
  配对前 - RNA: 2788, ATAC: 2737
RNA  barcode 示例: R1.33,R2.04,R3.35,P1.44
ATAC barcode 示例: R1.33,R2.01,R3.47,P1.36
RNA  格式: 逗号
ATAC 格式: 逗号

Cell key 示例:
  RNA:  R1.33,R2.04,R3.35,P1.44 → R1.33,R2.04,R3.35
  ATAC: R1.33,R2.01,R3.47,P1.36 → R1.33,R2.01,R3.47

RNA  duplicates: 34
ATAC duplicates: 43

RNA  cells: 2754
ATAC cells: 2694
Paired cells: 2060
配对成功！
  配对后: 2060 细胞

处理: GM12878.3T3.mm10
  配对前 - RNA: 3072, ATAC: 3035
RNA  barcode 示例: R1.33,R2.01,R3.47,P1.44
ATAC barcode 示例: R1.47,R2.55,R3.18,P1.31
RNA  格式: 逗号
ATAC 格式: 逗号

Cell key 示例:
  RNA:  R1.33,R2.01,R3.

# Extract Data

In [14]:
"""
case3_shareseq_analysis_v2.py
SHARE-seq 三层面分析（与 BMMC/AD/benchmark 互补）:
  层面一: 跨组织调控分歧 (skin vs brain vs lung)
  层面二: 增强子层面分析 (peak 组织特异性 + 调控距离)
  层面三: 非造血分化验证 (毛囊 vs BMMC 的分化模式对比)
"""

import os, sys
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from collections import Counter

######################################################################
# 参数
######################################################################

data_path = "/home/wuyan/dygmamba_project/NewRealPlan/case3/data/process/"
# 三组织路径
tissues = {
    "skin.late.anagen":  {"path": data_path + "skin/process/",  "color": "#E41A1C", "genome": "mm10"},
    "brain": {"path": data_path + "brain/process/", "color": "#377EB8", "genome": "mm10"},
    "lung":  {"path": data_path + "lung/process/",  "color": "#4DAF4A", "genome": "mm10"},
}
all_adata_rna = ad.read_h5ad(data_path + "merged_rna.h5ad")
all_adata_atac = ad.read_h5ad(data_path + "merged_atac.h5ad")

for cell_type, info in tissues.items():
    print(cell_type)
    print(info)
    cell_path = info['path']
    os.makedirs(cell_path, exist_ok=True)
    adata_rna = all_adata_rna[all_adata_rna.obs['cell_type']==cell_type].copy()
    adata_atac = all_adata_atac[all_adata_atac.obs['cell_type']==cell_type].copy()
    print(adata_rna)
    print(adata_atac)
    adata_rna.write_h5ad(cell_path + 'rna_origin.h5ad')
    adata_atac.write_h5ad(cell_path + 'atac_origin.h5ad')
    print(f"{cell_type} rna saved in {cell_path + 'rna_origin.h5ad'}")
    print(f"{cell_type} atac saved in {cell_path + 'atac_origin.h5ad'}")


skin.late.anagen
{'path': '/home/wuyan/dygmamba_project/NewRealPlan/case3/data/process/skin/process/', 'color': '#E41A1C', 'genome': 'mm10'}
AnnData object with n_obs × n_vars = 34054 × 51707
    obs: 'sample', 'cell_key', 'cell_type'
    layers: 'counts'
AnnData object with n_obs × n_vars = 34054 × 2182851
    obs: 'sample', 'cell_key', 'cell_type'
    layers: 'counts'
skin.late.anagen rna saved in /home/wuyan/dygmamba_project/NewRealPlan/case3/data/process/skin/process/rna_origin.h5ad
skin.late.anagen atac saved in /home/wuyan/dygmamba_project/NewRealPlan/case3/data/process/skin/process/atac_origin.h5ad
brain
{'path': '/home/wuyan/dygmamba_project/NewRealPlan/case3/data/process/brain/process/', 'color': '#377EB8', 'genome': 'mm10'}
AnnData object with n_obs × n_vars = 3291 × 51707
    obs: 'sample', 'cell_key', 'cell_type'
    layers: 'counts'
AnnData object with n_obs × n_vars = 3291 × 2182851
    obs: 'sample', 'cell_key', 'cell_type'
    layers: 'counts'
brain rna saved in /home/w

# Test

In [16]:
adata_rna.obs['cell_type']

lung_R1.41,R2.04,R3.01    lung
lung_R1.41,R2.04,R3.22    lung
lung_R1.41,R2.05,R3.78    lung
lung_R1.41,R2.06,R3.86    lung
lung_R1.41,R2.10,R3.06    lung
                          ... 
lung_R1.80,R2.84,R3.41    lung
lung_R1.80,R2.84,R3.45    lung
lung_R1.80,R2.85,R3.04    lung
lung_R1.80,R2.86,R3.71    lung
lung_R1.80,R2.93,R3.32    lung
Name: cell_type, Length: 1350, dtype: category
Categories (1, object): ['lung']